# Preprocessing pipeline — SisFall

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rokhaya8/fall-detection-lstm/blob/main/notebooks/01_preprocessing_sisfall.ipynb)

Part of [fall-detection-lstm](https://github.com/Rokhaya8/fall-detection-lstm). Paths are set by `BASE_DIR` in the first code cell — adapt it to your Google Drive.

 Setup : Montage du Drive et imports

Initialisation de l'environnement de travail. Deux options disponibles selon
votre accès aux données :
- **Option A** : accès au Drive partagé du groupe → exécuter directement

- **Option B** : utilisateurs externes → télécharger SisFall depuis Kaggle
  et adapter le chemin `SISFALL_ZIP`

In [ ]:
# OPTION A — Accès au Drive partagé
# ─────────────────────────────────────────────
# Ajouter le raccourci du dossier Projet_Parkinson dans votre Drive puis exécuter ce code
#
# OPTION B — Utilisateurs externes
# ──────────────────────────────────
# Télécharger SisFall depuis Kaggle :kaggle.com/datasets/nvnikhil0001/sis-fall-original-dataset
# Uploader sur votre Drive et adapter le chemin SISFALL_ZIP

from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

# ── Dossier du projet dans Google Drive (à adapter si besoin) ──
import os
BASE_DIR = '/content/drive/MyDrive/Projet_Parkinson'
os.makedirs(f'{BASE_DIR}/Figures', exist_ok=True)


# Chemin à adapter selon votre situation
SISFALL_ZIP = f'{BASE_DIR}/Datasets_bruts/SisFall_dataset.zip'
SISFALL_DIR = "/content/data/sisfall"

if not os.path.exists(SISFALL_DIR):
    print("Extraction SisFall...")
    os.makedirs(SISFALL_DIR, exist_ok=True)
    with zipfile.ZipFile(SISFALL_ZIP, 'r') as z:
        z.extractall(SISFALL_DIR)
    print("✓ SisFall prêt !")
else:
    print("✓ SisFall déjà extrait")


# ═══════════════════════════════════════════════
# IMPORTS — À exécuter en premier à chaque session
# ═══════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("✓ Tous les imports sont prêts !")

Mounted at /content/drive
Extraction SisFall...
✓ SisFall prêt !
✓ Tous les imports sont prêts !


Conversion physique : Unités brutes → Force en G

Les valeurs brutes de SisFall sont enregistrées en **bits**. Cette cellule
les convertit en **G** (force gravitationnelle) selon la formule officielle
du README SisFall :

**Accélération [g] = [(2 × Plage) / (2^Résolution)] × valeur_en_bits**

| Capteur | Plage | Résolution | Facteur de conversion |
|---|---|---|---|
| ADXL345 | ±16g | 13 bits | ≈ 0.003906 g/bit |
| MMA8451Q | ±8g | 14 bits | ≈ 0.000977 g/bit |

In [ ]:
def convertir_en_g(df):
    """
    Conversion des données de bits en unités physiques (g).

    Formules depuis le Readme officiel SisFall :
    Accélération [g] = [(2 × Range) / (2^Résolution)] × valeur_en_bits

    ADXL345  : Range=±16g, Résolution=13 bits
               facteur = (2×16) / (2^13) = 32/8192 ≈ 0.003906

    MMA8451Q : Range=±8g,  Résolution=14 bits
               facteur = (2×8)  / (2^14) = 16/16384 ≈ 0.000977

    Entrée : DataFrame en bits  (nb_lignes, 6)
    Sortie : DataFrame en g     (nb_lignes, 6)
    """

    df_converti = df.copy()

    # ── Conversion ADXL345 ────────────────────────────────────
    facteur_adxl = (2 * 16) / (2 ** 13)
    df_converti[['ADXL_X','ADXL_Y','ADXL_Z']] *= facteur_adxl

    # ── Conversion MMA8451Q ───────────────────────────────────
    facteur_mma = (2 * 8) / (2 ** 14)
    df_converti[['MMA_X','MMA_Y','MMA_Z']] *= facteur_mma

    return df_converti  # shape (nb_lignes, 6) en g

Nettoyage et chargement d'un fichier SisFall

Cette cellule définit la fonction de nettoyage appliquée à chaque fichier SisFall. Elle effectue les opérations suivantes dans l'ordre :

1. **Lecture** du fichier `.txt` brut
2. **Nettoyage** du caractère `;` parasite en colonne 8
3. **Sélection** des 6 colonnes accéléromètres (ADXL345 et MMA8451Q),
   les 3 colonnes du gyroscope ITG3200 sont écartées
4. **Conversion** en valeurs numériques
5. **Gestion des valeurs manquantes** par interpolation linéaire
6. **Conversion** bits → G (appel à la fonction `convertir_en_g`)
7. **Clipping** aux limites physiques des capteurs :
   - ADXL345 : [-16g, +16g]
   - MMA8451Q : [-8g, +8g]

In [ ]:
def nettoyer_fichier_sisfall(chemin_fichier):
    """
    ÉTAPE 1 — Chargement et nettoyage d'un fichier SisFall.

    Opérations :
    1. Lecture du fichier
    2. Nettoyage du ';' colonne 8
    3. Sélection des 6 colonnes accéléromètres
    4. Conversion en numérique
    5. Gestion des NA → interpolation linéaire
    6. Conversion bits → g
    7. Clipping aux limites physiques
    """

    # ── 1. Lecture ────────────────────────────────────────────
    df = pd.read_csv(chemin_fichier, header=None, sep=',')

    # ── 2. Nettoyage ';' colonne 8 ────────────────────────────
    df[8] = df[8].astype(str).str.replace(';', '').str.strip()
    df[8] = pd.to_numeric(df[8], errors='coerce')

    # ── 3. Sélection 6 colonnes ───────────────────────────────
    df_propre = df.iloc[:, [0,1,2,6,7,8]].copy()
    df_propre.columns = ['ADXL_X','ADXL_Y','ADXL_Z',
                          'MMA_X', 'MMA_Y', 'MMA_Z']

    # ── 4. Conversion en numérique ────────────────────────────
    df_propre = df_propre.apply(pd.to_numeric, errors='coerce')

    # ── 5. Gestion NA → interpolation ─────────────────────────
    nb_na = df_propre.isnull().sum().sum()
    if nb_na > 0:
        print(f"  {nb_na} NA dans "
              f"{os.path.basename(chemin_fichier)}"
              f" → interpolation")
        df_propre = df_propre.interpolate(method='linear')
        df_propre = df_propre.bfill().ffill()


    # ── 6. Conversion bits → g ────────────────────────────────
    df_propre = convertir_en_g(df_propre)

    # ── 7. Clipping limites physiques ─────────── GESTION DES OUTLIERS
    for col in ['ADXL_X','ADXL_Y','ADXL_Z']:
        df_propre[col] = df_propre[col].clip(-16, 16)
    for col in ['MMA_X','MMA_Y','MMA_Z']:
        df_propre[col] = df_propre[col].clip(-8, 8)

    return df_propre  # shape (nb_lignes, 6) en g

Cette cellule segmente les signaux continus en fenêtres de taille fixe,
prêtes à être ingérées par les modèles de deep learning.

**Paramètres de fenêtrage :**

| Paramètre | Valeur | Justification |
|---|---|---|
| Taille | 200 échantillons | = 1 seconde à 200 Hz, capture une chute complète |
| Pas | 100 échantillons | = 50% de chevauchement, standard dans la littérature |

**Étiquetage des fenêtres :**
Chaque fenêtre reçoit un label binaire selon la **règle majoritaire** :
- Label **1 (Chute)** si plus de 50% des échantillons de la fenêtre sont étiquetés "chute"
- Label **0 (Activité normale)** sinon

**Sortie :**
- `X` : tenseur de shape `(nb_fenêtres, 200, 6)`
- `y` : labels de shape `(nb_fenêtres,)`
- `sujets` : ID du sujet par fenêtre — conservé pour le split par sujet

In [ ]:
def creer_fenetres(données, labels, sujets,
                   taille=200, pas=100):
    """
    ÉTAPE 3 — Fenêtrage glissant.

    Entrée :
        données → tableau numpy shape (nb_échantillons, 6)
        labels  → tableau numpy shape (nb_échantillons,)
        sujets  → tableau numpy shape (nb_échantillons,)
                  contient l'ID du sujet pour chaque échantillon
                  → nécessaire pour le split par sujet
        taille  → 200 échantillons = 1.0 seconde à 200Hz
                  justification : capture une chute complète
        pas     → 100 = taille/2 = overlap 50%
                  justification : standard dans la littérature

    Sortie :
        X       → shape (nb_fenêtres, 200, 6)
        y       → shape (nb_fenêtres,)
        sujets  → shape (nb_fenêtres,)
                  ID du sujet pour chaque fenêtre
                  → utilisé pour le split par sujet

    Pourquoi on garde sujets ?
    → Le split par sujet nécessite de savoir
      à quel sujet appartient chaque fenêtre
    → Sans ça impossible de faire le split par sujet !
    """

    X, y, sujets_out = [], [], []

    nb_échantillons = len(données)

    for debut in range(0, nb_échantillons - taille + 1, pas):

        fin = debut + taille

        # Extraire la fenêtre de données (200 instants, 6 capteurs)
        fenetre_données = données[debut:fin]

        # Extraire les labels correspondants
        fenetre_labels  = labels[debut:fin]

        # Label majoritaire de la fenêtre
        label_fenetre = 1 if fenetre_labels.sum() > taille // 2 else 0

        # ID du sujet de la fenêtre
        # (tous les instants de la fenêtre ont le même sujet)
        sujet_fenetre = sujets[debut]

        X.append(fenetre_données)
        y.append(label_fenetre)
        sujets_out.append(sujet_fenetre)

    return np.array(X), np.array(y), np.array(sujets_out)

Cette cellule divise les données en trois ensembles en respectant
le **principe du split par sujet** : un sujet entier appartient
à un seul groupe, jamais réparti entre plusieurs.

**Pourquoi un split par sujet ?**
Un split aléatoire classique introduirait du **data leakage** — des
fenêtres du même sujet se retrouveraient à la fois dans le train et
le test, rendant l'évaluation trop optimiste. Le split par sujet
simule la généralisation sur de nouveaux patients jamais vus.

**Répartition des sujets :**

| Ensemble | Sujets | Proportion |
|---|---|---|
| Train | 29 sujets (SA01→SA18, SE01→SE11) | ~70% |
| Validation | SA17, SA18, SE12, SE13 | ~15% |
| Test | SA19, SA20, SA21, SE14, SE15 | ~15% |

Chaque groupe contient un mix de **jeunes adultes (SA)** et de
**personnes âgées (SE)** pour approximer une distribution représentative.

**Vérifications effectuées :**
- Proportions train/val/test
- Ratio chutes/activités dans chaque groupe
- Avertissement si un groupe est très déséquilibré (ratio < 20% ou > 60%)

In [ ]:
def splitter_par_sujet(X, y, sujets):
    """
    ÉTAPE 4 — Split par sujet train / val / test.

    Principe :
    → Un sujet entier va dans UN SEUL groupe
    → Jamais le même sujet dans deux groupes
    → Élimine tout data leakage
    → Simule la généralisation sur nouveaux patients

    Note sur la stratification :
    → Avec le split par sujet, on ne peut pas
      garantir exactement le même ratio chutes/activités
      dans chaque groupe
    → On choisit les sujets avec un mix adultes/âgés
      pour approximer une distribution équilibrée
    → On vérifie le ratio après le split

    Entrée :
        X      → shape (nb_fenêtres, 200, 6)
        y      → shape (nb_fenêtres,)
        sujets → shape (nb_fenêtres,)

    Sortie :
        X_train, X_val, X_test
        y_train, y_val, y_test
    """

    # ── Définition des sujets ─────────────────────────────────
    sujets_test  = ['SA19', 'SA20', 'SA21', 'SE14', 'SE15']
    sujets_val   = ['SA17', 'SA18', 'SE12', 'SE13']
    sujets_train = [s for s in np.unique(sujets)
                    if s not in sujets_test
                    and s not in sujets_val]

    # ── Masques booléens ──────────────────────────────────────
    masque_train = np.isin(sujets, sujets_train)
    masque_val   = np.isin(sujets, sujets_val)
    masque_test  = np.isin(sujets, sujets_test)

    # ── Application ───────────────────────────────────────────
    X_train = X[masque_train]
    X_val   = X[masque_val]
    X_test  = X[masque_test]
    y_train = y[masque_train]
    y_val   = y[masque_val]
    y_test  = y[masque_test]

    # ── Bilan ─────────────────────────────────────────────────
    total = len(X)
    print("=== SPLIT PAR SUJET ===")
    print(f"\nSujets train ({len(sujets_train)}) : {sujets_train}")
    print(f"Sujets val   ({len(sujets_val)})  : {sujets_val}")
    print(f"Sujets test  ({len(sujets_test)}) : {sujets_test}")

    print(f"\nProportions :")
    print(f"  Train : {len(X_train)/total*100:.1f}% "
          f"({len(X_train)} fenêtres)")
    print(f"  Val   : {len(X_val)/total*100:.1f}%   "
          f"({len(X_val)} fenêtres)")
    print(f"  Test  : {len(X_test)/total*100:.1f}%  "
          f"({len(X_test)} fenêtres)")

    # ── Vérification ratio chutes/activités ───────────────────
    print(f"\nRatio chutes/activités :")
    print(f"  Train → chutes {y_train.mean()*100:.1f}% "
          f"| activités {(1-y_train.mean())*100:.1f}%")
    print(f"  Val   → chutes {y_val.mean()*100:.1f}%   "
          f"| activités {(1-y_val.mean())*100:.1f}%")
    print(f"  Test  → chutes {y_test.mean()*100:.1f}%  "
          f"| activités {(1-y_test.mean())*100:.1f}%")

    # ── Avertissement si ratio très déséquilibré ──────────────
    for nom, y_g in [('Train', y_train),
                      ('Val',   y_val),
                      ('Test',  y_test)]:
        ratio = y_g.mean()
        if ratio < 0.20 or ratio > 0.60:
            print(f"\n  Attention : ratio {nom} "
                  f"({ratio*100:.1f}%) très déséquilibré !")

    return X_train, X_val, X_test, \
           y_train, y_val, y_test

Cette cellule normalise les données en appliquant une standardisation
Z-score : **X_scaled = (X - mean) / std**

**Règle fondamentale — éviter le data leakage :**
- `fit()` uniquement sur **X_train** → le scaler apprend la moyenne et
  l'écart-type des données d'entraînement uniquement
- `transform()` appliqué sur train, val **et** test
- Jamais de `fit()` sur val ou test → ce serait du data leakage

**Pourquoi StandardScaler plutôt que MinMaxScaler ?**

| Critère | StandardScaler | MinMaxScaler |
|---|---|---|
| Robustesse aux outliers | Robuste aux pics de chute |  Sensible aux valeurs extrêmes |
| Applicable à Daphnet | Même scaler réutilisable | Bornes dépendantes des données |
| Adapté aux LSTM/GRU/CNN | Standard | Standard |

**Pourquoi le reshape 3D → 2D ?**
StandardScaler travaille uniquement en 2D. Les tenseurs sont donc
temporairement aplatis de `(nb_fenêtres, 200, 6)` vers
`(nb_fenêtres × 200, 6)` puis remis en 3D après normalisation.

In [ ]:
def standardiser(X_train, X_val, X_test):
    """
    ÉTAPE 5 — Standardisation des données (Z-score).

    Formule : X_scaled = (X - mean) / std

    Entrée :
        X_train → shape (nb_fenêtres, 200, 6)
        X_val   → shape (nb_fenêtres, 200, 6)
        X_test  → shape (nb_fenêtres, 200, 6)

    Sortie :
        X_train, X_val, X_test standardisés
        scaler → à sauvegarder pour Daphnet !

    RÈGLE FONDAMENTALE :
    → fit() UNIQUEMENT sur X_train
    → transform() sur train, val ET test
    → Jamais fit() sur val ou test
      → ce serait du data leakage !

    Pourquoi StandardScaler et pas MinMax ?
    → Robuste aux pics de chute (outliers)
    → Même scaler applicable à Daphnet
    → Standard pour LSTM, GRU et CNN

    Pourquoi reshape en 2D ?
    → StandardScaler travaille en 2D uniquement
    → On reshape (nb_fenêtres, 200, 6)
      vers (nb_fenêtres × 200, 6)
    → On remet en 3D après
    """

    nb_train    = X_train.shape[0]
    nb_val      = X_val.shape[0]
    nb_test     = X_test.shape[0]
    taille      = X_train.shape[1]   # 200
    nb_features = X_train.shape[2]   # 6

    # ── Reshape 3D → 2D ───────────────────────────────────────
    X_train_2d = X_train.reshape(-1, nb_features)
    X_val_2d   = X_val.reshape(-1, nb_features)
    X_test_2d  = X_test.reshape(-1, nb_features)

    # ── Ajuster le scaler sur train UNIQUEMENT ────────────────
    scaler = StandardScaler()
    scaler.fit(X_train_2d)

    # ── Appliquer à train, val et test ────────────────────────
    X_train_scaled = scaler.transform(X_train_2d)
    X_val_scaled   = scaler.transform(X_val_2d)
    X_test_scaled  = scaler.transform(X_test_2d)

    # ── Reshape 2D → 3D ───────────────────────────────────────
    X_train_scaled = X_train_scaled.reshape(
        nb_train, taille, nb_features)
    X_val_scaled   = X_val_scaled.reshape(
        nb_val, taille, nb_features)
    X_test_scaled  = X_test_scaled.reshape(
        nb_test, taille, nb_features)

    # ── Vérification ──────────────────────────────────────────
    print("=== STANDARDISATION ===")
    print(f"Moyenne avant : {X_train_2d.mean():.4f}")
    print(f"Std avant     : {X_train_2d.std():.4f}")
    print(f"Moyenne après : "
          f"{X_train_scaled.reshape(-1,nb_features).mean():.4f}"
          f" (attendu ≈ 0)")
    print(f"Std après     : "
          f"{X_train_scaled.reshape(-1,nb_features).std():.4f}"
          f" (attendu ≈ 1)")
    print(f"\nShape X_train : {X_train_scaled.shape}")
    print(f"Shape X_val   : {X_val_scaled.shape}")
    print(f"Shape X_test  : {X_test_scaled.shape}")
    print(f"\nMoyennes apprises : {scaler.mean_.round(4)}")
    print(f"Std apprises      : {scaler.scale_.round(4)}")

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

Cette cellule sauvegarde les tenseurs préprocessés et le scaler
sur Google Drive pour un accès persistant entre les sessions Colab.

**Fichiers sauvegardés :**

| Fichier | Contenu | Format |
|---|---|---|
| `X_train.npy` | Données d'entraînement | NumPy |
| `X_val.npy` | Données de validation | NumPy |
| `X_test.npy` | Données de test | NumPy |
| `y_train.npy` | Labels d'entraînement | NumPy |
| `y_val.npy` | Labels de validation | NumPy |
| `y_test.npy` | Labels de test | NumPy |
| `scaler.pkl` | StandardScaler ajusté | Pickle |

**Pourquoi le format `.npy` ?**

Format natif NumPy — rapide à charger, conserve exactement
la shape et le dtype, plus léger que `.csv` ou `.json`.

In [ ]:
def sauvegarder(X_train, X_val, X_test,
                y_train, y_val, y_test,
                scaler,
                chemin=f'{BASE_DIR}/Data_Preprocessed/SisFall'):
    """
    ÉTAPE 6 — Sauvegarde des données préprocessées.

    Fichiers sauvegardés :
    → X_train.npy, X_val.npy, X_test.npy  (données)
    → y_train.npy, y_val.npy, y_test.npy  (labels)
    → scaler.pkl                           (StandardScaler)

    Pourquoi .npy ?
    → Format natif NumPy
    → Rapide à charger
    → Conserve exactement shape et dtype
    → Léger comparé à .csv ou .json

    Pourquoi sauvegarder le scaler ?
    → Doit être réutilisé sur Daphnet
    → Sans ça les données ne sont plus
      dans la même échelle

    Pourquoi sauvegarder sur Drive ?
    → Persist après reset Colab
    → Accessible à tout le groupe
    → Pas besoin de relancer la pipeline
    """

    # ── Créer le dossier si inexistant ────────────────────────
    os.makedirs(chemin, exist_ok=True)

    # ── Sauvegarder les tableaux NumPy ────────────────────────
    np.save(f'{chemin}/X_train.npy', X_train)
    np.save(f'{chemin}/X_val.npy',   X_val)
    np.save(f'{chemin}/X_test.npy',  X_test)
    np.save(f'{chemin}/y_train.npy', y_train)
    np.save(f'{chemin}/y_val.npy',   y_val)
    np.save(f'{chemin}/y_test.npy',  y_test)

    # ── Sauvegarder le scaler ─────────────────────────────────
    with open(f'{chemin}/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    # ── Vérification ──────────────────────────────────────────
    print("=== SAUVEGARDE ===")
    print(f"Dossier : {chemin}")
    print(f"\nFichiers sauvegardés :")
    print(f"  X_train.npy → {X_train.shape}")
    print(f"  X_val.npy   → {X_val.shape}")
    print(f"  X_test.npy  → {X_test.shape}")
    print(f"  y_train.npy → {y_train.shape}")
    print(f"  y_val.npy   → {y_val.shape}")
    print(f"  y_test.npy  → {y_test.shape}")
    print(f"  scaler.pkl  → StandardScaler sauvegardé")

    # ── Taille des fichiers ───────────────────────────────────
    total_mb = 0
    for fichier in ['X_train.npy', 'X_val.npy', 'X_test.npy',
                    'y_train.npy', 'y_val.npy', 'y_test.npy']:
        taille = os.path.getsize(
            f'{chemin}/{fichier}') / (1024*1024)
        total_mb += taille
        print(f"  {fichier} : {taille:.2f} MB")
    print(f"\nTotal : {total_mb:.2f} MB")
    print("\nSauvegarde terminée ✓")

Cette cellule orchestre l'ensemble du preprocessing SisFall en appelant
toutes les fonctions précédentes dans l'ordre correct.

**Étapes exécutées :**

| Étape | Description |
|---|---|
| 1 | Chargement, nettoyage et conversion de tous les fichiers |
| 2 | Fenêtrage glissant (taille=200, pas=100) |
| 3 | Split par sujet (train / validation / test) |
| 4 | Standardisation (StandardScaler ajusté sur train) |
| 5 | Sauvegarde des tenseurs et du scaler sur Drive |

**Paramètres fixés par le groupe :**

| Paramètre | Valeur | Justification |
|---|---|---|
| `taille_fenetre` | 200 | 1 seconde à 200 Hz |
| `pas` | 100 | Overlap 50% |
| Split | Par sujet | Pas de data leakage |
| Standarisation | StandardScaler | Robuste aux pics de chute |

**Sortie :**
- `X_train`, `X_val`, `X_test` → shape `(nb_fenêtres, 200, 6)`
- `y_train`, `y_val`, `y_test` → shape `(nb_fenêtres,)`
- `scaler` → StandardScaler à réutiliser sur Daphnet

In [ ]:
def pipeline_sisfall(chemin_sisfall,
                     chemin_sauvegarde=f'{BASE_DIR}/Data_Preprocessed/SisFall',
                     taille_fenetre=200,
                     pas=100):
    """
    FONCTION PRINCIPALE — Pipeline complète préprocessing SisFall.

    Appelle toutes les étapes dans l'ordre :
    1. Chargement + nettoyage + NA + conversion + clipping
    2. Fenêtrage glissant (taille=200, pas=100)
    3. Split par sujet (train/val/test)
    4. Standardisation (StandardScaler sur train)
    5. Sauvegarde (.npy + scaler.pkl)

    Paramètres fixés par le groupe :
    → taille_fenetre = 200  (1.0 seconde à 200Hz)
    → pas            = 100  (overlap 50%)
    → Split par sujet       (pas de data leakage)
    → StandardScaler        (robuste aux pics de chute)

    Entrée :
        chemin_sisfall    → dossier contenant SisFall
        chemin_sauvegarde → dossier Drive pour sauvegarder
        taille_fenetre    → taille fenêtre (défaut=200)
        pas               → pas fenêtrage  (défaut=100)

    Sortie :
        X_train, X_val, X_test → shape (nb_fenêtres, 200, 6)
        y_train, y_val, y_test → shape (nb_fenêtres,)
        scaler                 → StandardScaler ajusté sur train
    """

    print("=" * 55)
    print("   PIPELINE PRÉPROCESSING SISFALL")
    print("=" * 55)

    # ── Initialisation ────────────────────────────────────────
    toutes_donnees = []
    tous_labels    = []
    tous_sujets    = []
    nb_fichiers    = 0
    nb_chutes      = 0
    nb_activites   = 0

    # ── ÉTAPES 1 : Charger, nettoyer, convertir ─────────────
    print("\n[1/6] Chargement, nettoyage et conversion...")

    # Recherche récursive des fichiers : fonctionne quelle que soit
    # la structure interne du zip téléchargé (Kaggle, site officiel...)
    fichiers_trouves = {}
    for racine, _, fichiers in os.walk(chemin_sisfall):
        for fichier in fichiers:
            if fichier.endswith('.txt') and fichier.startswith(('F', 'D')):
                # Clé = (dossier du sujet, nom du fichier) → même ordre
                # de lecture qu'avant ; les doublons éventuels sont ignorés
                cle = (os.path.basename(racine), fichier)
                fichiers_trouves.setdefault(cle, os.path.join(racine, fichier))

    for (_, fichier), chemin_fichier in sorted(fichiers_trouves.items()):
        # Extraire label et sujet depuis le nom du fichier
        if fichier.startswith('F'):
            label = 1
            nb_chutes += 1
        elif fichier.startswith('D'):
            label = 0
            nb_activites += 1
        else:
            continue

        # Extraire l'ID du sujet
        # ex: F01_SA01_R01.txt → SA01
        try:
            sujet_id = fichier.split('_')[1]
        except IndexError:
            continue


        try:
            # Étape 1 — Nettoyer
            df = nettoyer_fichier_sisfall(chemin_fichier)

            # Convertir en numpy
            donnees  = df.values
            labels   = np.full(len(donnees), label)
            sujets   = np.array([sujet_id] * len(donnees))

            toutes_donnees.append(donnees)
            tous_labels.append(labels)
            tous_sujets.append(sujets)
            nb_fichiers += 1

        except Exception as e:
            print(f"  Fichier ignoré : {fichier} → {e}")

    print(f"  {nb_fichiers} fichiers chargés "
          f"({nb_chutes} chutes, {nb_activites} activités)")

    # Concatener tout
    donnees_concat = np.concatenate(toutes_donnees, axis=0)
    labels_concat  = np.concatenate(tous_labels,    axis=0)
    sujets_concat  = np.concatenate(tous_sujets,    axis=0)

    print(f"  Shape totale : {donnees_concat.shape}")

    # ── ÉTAPE 2 : Fenêtrage glissant ──────────────────────────
    print("\n[2/6] Fenêtrage glissant...")
    X, y, sujets_f = creer_fenetres(
        donnees_concat, labels_concat, sujets_concat,
        taille_fenetre, pas)
    print(f"  X shape : {X.shape}")
    print(f"  Chutes  : {y.sum()} ({y.mean()*100:.1f}%)")
    print(f"  Activités : {(y==0).sum()} "
          f"({(y==0).mean()*100:.1f}%)")

    # ── ÉTAPE 3 : Split par sujet ─────────────────────────────
    print("\n[3/6] Split par sujet...")
    X_train, X_val, X_test, \
    y_train, y_val, y_test = splitter_par_sujet(X, y, sujets_f)

    # ── ÉTAPE 4 : Standardisation ─────────────────────────────
    print("\n[4/6] Standardisation...")
    X_train, X_val, X_test, scaler = standardiser(
        X_train, X_val, X_test)

    # ── ÉTAPE 5 : Sauvegarde ──────────────────────────────────
    print("\n[5/6] Sauvegarde...")
    sauvegarder(X_train, X_val, X_test,
                y_train, y_val, y_test,
                scaler, chemin_sauvegarde)

    # ── Résumé final ──────────────────────────────────────────
    print("\n" + "=" * 55)
    print("   PIPELINE TERMINÉE AVEC SUCCÈS !")
    print("=" * 55)
    print(f"\nRésumé final :")
    print(f"  X_train : {X_train.shape} | "
          f"Chutes : {y_train.mean()*100:.1f}%")
    print(f"  X_val   : {X_val.shape} | "
          f"Chutes : {y_val.mean()*100:.1f}%")
    print(f"  X_test  : {X_test.shape} | "
          f"Chutes : {y_test.mean()*100:.1f}%")
    print(f"\nScaler sauvegardé → à réutiliser sur Daphnet !")

    return X_train, X_val, X_test, \
           y_train, y_val, y_test, scaler

Exécution de la pipeline SisFall
Lance l'ensemble du preprocessing SisFall en appelant `pipeline_sisfall()`.
Les tenseurs préprocessés et le scaler sont sauvegardés automatiquement
sur Google Drive à la fin de l'exécution.


In [ ]:
X_train, X_val, X_test, \
y_train, y_val, y_test, scaler = pipeline_sisfall(
    chemin_sisfall    = SISFALL_DIR,  # dossier d'extraction du zip
    chemin_sauvegarde = f'{BASE_DIR}/Data_Preprocessed/SisFall',
    taille_fenetre    = 200,
    pas               = 100
)

   PIPELINE PRÉPROCESSING SISFALL

[1/6] Chargement, nettoyage et conversion...
  4505 fichiers chargés (1798 chutes, 2707 activités)
  Shape totale : (15858929, 6)

[2/6] Fenêtrage glissant...
  X shape : (158588, 200, 6)
  Chutes  : 53937 (34.0%)
  Activités : 104651 (66.0%)

[3/6] Split par sujet...
=== SPLIT PAR SUJET ===

Sujets train (29) : [np.str_('SA01'), np.str_('SA02'), np.str_('SA03'), np.str_('SA04'), np.str_('SA05'), np.str_('SA06'), np.str_('SA07'), np.str_('SA08'), np.str_('SA09'), np.str_('SA10'), np.str_('SA11'), np.str_('SA12'), np.str_('SA13'), np.str_('SA14'), np.str_('SA15'), np.str_('SA16'), np.str_('SA22'), np.str_('SA23'), np.str_('SE01'), np.str_('SE02'), np.str_('SE03'), np.str_('SE04'), np.str_('SE05'), np.str_('SE06'), np.str_('SE07'), np.str_('SE08'), np.str_('SE09'), np.str_('SE10'), np.str_('SE11')]
Sujets val   (4)  : ['SA17', 'SA18', 'SE12', 'SE13']
Sujets test  (5) : ['SA19', 'SA20', 'SA21', 'SE14', 'SE15']

Proportions :
  Train : 77.6% (123031 fenêt